The same lab as a web page, with the folded answers: https://ml.wxing.me/materials/lesson6/lab.html

# Lab 6: Grow a Tree, Choose Its Depth, Read It
## MPS311/439 Machine Learning
**Lesson 6 Lab Session | Core: about 40 minutes**

## What this lab is for

All the code is given. You run it, change the value marked `← change`, and look at what happens. After each Part, **write down** one number from your output and one sentence. The folded **Challenge** and **Try with AI** boxes are for when you finish early; the **Advanced Study** is for after class.

## How to work

1. This notebook already has all the code. Save a copy first (**File → Save a copy in Drive**). Run the cells from top to bottom: later cells use names made by earlier ones.
2. Write your answers in a text cell (**+ Text**) under the code.
3. **Run, and you should see** tells you what your output should show, so you know you are on track.

## Part 0: Set up and grow a first tree (6 minutes)

Run this cell without reading it closely. It is the same at the start of every lab.

In [ ]:
import numpy as np                                        # numbers and arrays
import pandas as pd                                       # tables
import matplotlib.pyplot as plt                           # plots
from sklearn.datasets import load_iris                    # today's dataset
from sklearn.model_selection import train_test_split, KFold, cross_val_score, cross_val_predict   # splitting and cross-validation
from sklearn.preprocessing import StandardScaler          # puts every column on one scale
from sklearn.tree import DecisionTreeClassifier, plot_tree   # today's model, and a picture of it
from sklearn.linear_model import LogisticRegression       # a straight-line classifier, for comparison

**The data.** 150 iris flowers of three species (0 setosa, 1 versicolor, 2 virginica), with four measurements in cm: sepal length, sepal width, petal length, petal width. You do not change anything in this block today.

In [ ]:
data = load_iris()                                        # ← change: which dataset (not today)
X = data.data                                             # the measurements (features)
y = data.target                                           # ← change: the labels (not today; here the species 0, 1, 2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)     # 3:1 split, the same mix of labels in both parts
scaler = StandardScaler()                                 # one scale for every column
X_train_s = scaler.fit_transform(X_train)                 # learn the scale from the training data only
X_test_s = scaler.transform(X_test)                       # apply the same scale to the test data
kf = KFold(5, shuffle=True, random_state=0)               # the same 5 folds every time
print(X_train.shape, np.bincount(y_train))                # training rows and columns; how many of each label

**Run, and you should see:** `(112, 4) [37 37 38]`.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # a tree that asks at most 2 questions
print(cross_val_score(tree, X_train, y_train, cv=kf).mean())  # 5-fold cross-validated accuracy

**Run, and you should see:** about `0.955`. A tree does not need the standardised `X_train_s`: each question compares one measurement with one number.

## Part 1: Read the tree (8 minutes)

In [ ]:
names = ["sepal length", "sepal width", "petal length", "petal width"]   # the four columns, in order
tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # the tree of Part 0
tree.fit(X_train, y_train)                                # fit on all 112 training flowers
plt.figure(figsize=(14, 7))                               # a large figure, so the boxes are readable
plot_tree(tree, feature_names=names, class_names=["setosa", "versicolor", "virginica"], filled=True, fontsize=12)   # draw it
plt.show()

Each box shows its question (leaves have none), its Gini, `samples` (how many training flowers reach it) and `value` (how many of each species). Follow the arrows: left is "yes", right is "no".

**Write down:** which measurement the first question asks about, and the `value` of the leaf that mixes two species. A flower landing in that leaf: what probability does the tree give for versicolor?

*Your answer:* 

> **Challenge (if you finish early)**
>
> Add `criterion="entropy"` inside `DecisionTreeClassifier(...)` and draw the tree again. Is the first question the same? Does the cross-validated accuracy of Part 0 change?

## Part 2: How deep should the tree go? (15 minutes)

With all four measurements the species are easy to tell apart, so a deep tree hardly does any harm. To see what goes wrong, use only the two **sepal** measurements, where versicolor and virginica overlap.

In [ ]:
X_sepal = X_train[:, :2]                                  # only columns 0 and 1: sepal length and width
tree = DecisionTreeClassifier(max_depth=None, random_state=0)   # ← change: None means no depth limit
tree.fit(X_sepal, y_train)                                # grow the tree on all 112 training flowers
print(tree.get_n_leaves(), tree.score(X_sepal, y_train))  # number of leaves, training accuracy
print(cross_val_score(tree, X_sepal, y_train, cv=kf).mean())    # 5-fold cross-validated accuracy

**Run, and you should see:** `37` leaves, training accuracy about `0.946`, cross-validated accuracy about `0.696`. Then change `None` to `3` and run again.

Now try many depths at once and draw both accuracies:

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10]                        # ← change: the depths to try
train_list = []                                           # training accuracy, one per depth
cv_list = []                                              # cross-validated accuracy, one per depth
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_sepal, y_train)                            # fit on all training flowers
    train_list.append(tree.score(X_sepal, y_train))       # accuracy on the flowers it was fitted on
    cv_list.append(cross_val_score(tree, X_sepal, y_train, cv=kf).mean())   # accuracy on flowers it did not see
    print(depth, round(train_list[-1], 3), round(cv_list[-1], 3))
plt.plot(depths, train_list, "o-", label="training")      # one line per kind of accuracy
plt.plot(depths, cv_list, "s-", label="5-fold cross-validation")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()

**Write down:** the depth with the highest cross-validated accuracy (if two tie, take the shallower one), and whether it sits at an end of `depths`. In one sentence: why is the training line so much higher than the cross-validated line for deep trees?

*Your answer:* 

**Think about** (with your neighbour, no writing): a tree scores 100% on its training data. Is that good news?

> **Try with AI (if you finish early, or after class)**
>
> 1. **You want:** a picture of the boxes the tree cuts in the sepal plane, for a deep tree and a shallow one.
> 2. **Tell the AI:** `X_sepal`, `y_train`, and two trees: `DecisionTreeClassifier(max_depth=None, random_state=0)` and `DecisionTreeClassifier(max_depth=3, random_state=0)`.
> 3. **Ask:** "Write simple Python that fits each tree on `X_sepal`, `y_train` and draws its decision regions with `sklearn.inspection.DecisionBoundaryDisplay`, with the training flowers on top coloured by species, side by side."
> 4. **Check:** the deep tree's picture has many small boxes, some around one or two flowers; the depth-3 tree has a few large boxes, all with straight horizontal and vertical edges.

> **Challenge (if you finish early)**
>
> `max_depth` is one way to stop a tree. Another is `min_samples_leaf`: every leaf must keep at least that many flowers. In the first cell of this Part, replace `max_depth=None` with `min_samples_leaf=5`, then try `10`, `20` and `30`. Which value is best, and why does `30` suddenly fall so low? (Count the flowers of each species.)

## Part 3: Which measurements does the tree use? (8 minutes)

Back to all four measurements.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=0)   # a tree that asks at most 3 questions
tree.fit(X_train, y_train)                                # fit on all training flowers
print(pd.Series(tree.feature_importances_, index=names).round(3))   # importance: each column's share of the Gini drop

**Run, and you should see:** petal width `0.968`, petal length `0.032`, the two sepal measurements `0.0`. The importances add up to 1.

**Write down:** the measurement the tree relies on most, with its importance.

*Your answer:* 

> **Challenge: is petal length useless? (if you finish early)**
>
> Petal length scores only 0.032. Take petal width away and grow the same tree again:

In [ ]:
keep = [0, 1, 2]                                          # ← change: the columns to keep (here: all but petal width, column 3)
X_less = X_train[:, keep]                                 # the training flowers with only these columns
tree_less = DecisionTreeClassifier(max_depth=3, random_state=0)   # the same kind of tree as above
print(cross_val_score(tree_less, X_less, y_train, cv=kf).mean())   # the same 5 folds as before
tree_less.fit(X_less, y_train)                            # fit without the removed column
print(pd.Series(tree_less.feature_importances_, index=np.array(names)[keep]).round(3))
print(np.corrcoef(X_train[:, 2], X_train[:, 3])[0, 1])    # correlation of petal length and petal width

> What happens to the accuracy and to petal length's importance? The two petal measurements have correlation 0.964. Does an importance near 0 mean a measurement is useless?
> 
> *One possible answer:* without petal width, petal length takes over (0.986) and the accuracy is still 0.892. The two petal measurements say almost the same thing, so the tree picks one and the other gets little credit. An importance near 0 says only that this tree did not need the column, given the others.

> **Challenge: how sure is the tree? (if you finish early)**
>
> A leaf's shares are the tree's probabilities. This cell gives every training flower the probabilities from a tree that did not see it, and counts the flowers whose highest probability is below 0.95 (our own choice of "not sure"):

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # ← change: try None
proba = cross_val_predict(tree, X_train, y_train, cv=kf, method="predict_proba")   # 3 probabilities per flower
print("not sure (top probability below 0.95):", (proba.max(axis=1) < 0.95).sum())
print("wrong:", (proba.argmax(axis=1) != y_train).sum())

> Run it at depth 2, then with `max_depth=None`. With no limit every flower looks sure, yet the tree is wrong more often. Why can you not trust a deep tree's probabilities?

## Part 4: The test set, once (3 minutes)

Choose the depth you would use for all four measurements (Part 0 gave 0.955 at depth 2). Then read the test set once.

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0)   # ← change: the depth you choose
tree.fit(X_train, y_train)                                # fit on all 112 training flowers
print(tree.score(X_test, y_test))                         # the 38 test flowers, read once

**Run, and you should see:** about `0.947` at depth 2. Whatever you chose, do not change it after seeing this number: that would be choosing on the test set.

# After class

## Choose the depth with one validation set (for Assignment 2)

Assignment 2, Section 2 chooses `max_depth` on one validation set, not by cross-validation. Here is the same idea on the sepal measurements: hold out 28 of the 112 flowers, fit on the other 84, and score each depth on the 28.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_sepal, y_train, test_size=0.25, random_state=0, stratify=y_train)   # one validation set of 28 flowers
for depth in [1, 2, 3, 4, 5, 6, 7, 8]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0)
    tree.fit(X_fit, y_fit)                                # fit on the 84 fitting flowers
    print(depth, round(tree.score(X_val, y_val), 3))      # accuracy on the 28 validation flowers

This set picks depth `4`; with `random_state=1` and `2` the picks are `5` and `3`. With only 28 validation flowers, one flower is 0.036 of the accuracy, so the pick depends on luck. In Assignment 2 the validation set has 10,000 images, one image is 0.0001, and one validation set is enough. Use this loop there with the assignment's `X_train`, `y_train`, `X_val`, `y_val`.

## For higher marks: a small investigation on the Titanic passengers

Optional. The Note grew its trees on the Titanic passengers. Load them with the same data block (two lines of cleaning are new):

In [ ]:
ship = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv")   # 891 passengers
ship = ship[["survived", "pclass", "sex", "age", "sibsp", "parch", "fare"]].dropna()   # 6 features and the label; keep the 714 with a known age
ship["sex"] = (ship["sex"] == "female").astype(int)       # 1 = female, 0 = male
X = ship.drop(columns="survived").values                  # ← change: the 6 features
y = ship["survived"].values                               # ← change: 1 = survived, 0 = died
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
kf = KFold(5, shuffle=True, random_state=0)
print(X_train.shape, np.bincount(y_train))

**Run, and you should see:** `(535, 6) [318 217]`.

In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=0)   # the depth chosen in the Note
print(cross_val_score(tree, X_train, y_train, cv=kf).mean())
line = LogisticRegression(max_iter=1000)                  # a straight line, on the standardised features
print(cross_val_score(line, X_train_s, y_train, cv=kf).mean())

Which method would you choose for these passengers, why, and when would you change your mind? Use at least one more run of your own as evidence (another depth, a feature removed, `plot_tree`). Section 8 of the Note shows one way to write it up: observe, idea, action, test, explain, limits.

## Reflection

One or two sentences each.

1. What does a decision tree's `.fit()` actually do? What does it try, and what does it keep?
2. A tree scores 100% on its training data and 75% under cross-validation. Is it a good model? What would you change?
3. When would you choose a decision tree over logistic regression, and when not?

## How today's lab helps with Assignment 2

| What you did today | Where it appears in Assignment 2 |
|---|---|
| Part 2: try several `max_depth` values; the best must not sit at an end | Section 2: "Choose both using the validation set only ... The chosen value must not lie at either end of your search range" |
| After class: choose the depth on one validation set | Section 2: use the same loop with the assignment's `X_train`, `y_train`, `X_val`, `y_val` |
| Part 2: training and cross-validated accuracy against depth | Section 2, Going further: the figure of training and validation accuracy against depth |
| Part 4: read the test set once | Section 4: `test` is used once, at the end |